## Setup

Before running: set two environment variables (Kaggle/Colab secrets, or
`os.environ[...] = "..."` in a cell you don't share) —

- `WANDB_API_KEY`
- `HF_TOKEN` (needs `write` scope to push the adapter to your HF account)

If you previously had a WandB key or HF token hardcoded anywhere, revoke
both on wandb.ai/authorize and huggingface.co/settings/tokens — they were
exposed in plaintext.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"


In [ ]:
!pip install wandb


## Detect GPU precision support

Everything downstream (`dtype` on model load, `fp16`/`bf16` on the
trainer) is derived from this single check, so the model's weight
dtype and the trainer's mixed-precision mode always match. Mismatching
these two is what caused
`"_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'`
— GradScaler only supports fp16 tensors, so if the trainer is in fp16
mode but the model weights are bf16 (or vice versa on non-bf16
hardware), it crashes.

In [ ]:
import torch

BF16_OK = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if BF16_OK else torch.float16

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"bf16 supported: {BF16_OK}")
print(f"Using dtype: {COMPUTE_DTYPE}")


## Load model and tokenizer

`dtype` is pinned explicitly to `COMPUTE_DTYPE` so the loaded weights
always match the precision the trainer will use below.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-0.6B",
    max_seq_length=2048,
    dtype=COMPUTE_DTYPE,
    load_in_4bit=True,
)


## Load and format GSM8K

Train and eval go through the *same* conversion pipeline: strip
`<<...>>` calculator annotations, convert to OpenAI-style
`role`/`content` messages (what `tokenizer.apply_chat_template`
expects for Qwen), then render to a flat `"text"` field via the chat
template. Same pipeline for both splits avoids any schema drift
between train and eval.

In [ ]:
import re
from datasets import load_dataset

# 1. Load GSM8K train + test
train_raw = load_dataset("openai/gsm8k", "main", split="train")
eval_raw  = load_dataset("openai/gsm8k", "main", split="test")

# 2. Strip <<...>> calculator annotations, keep #### final answer
def clean_answer(answer_text):
    return re.sub(r"<<.*?>>", "", answer_text).strip()

def to_conversations(example):
    return {
        "conversations": [
            {"role": "user", "content": example["question"]},
            {"role": "assistant", "content": clean_answer(example["answer"])},
        ]
    }

train_ds = train_raw.map(to_conversations, remove_columns=["question", "answer"])
eval_ds  = eval_raw.map(to_conversations, remove_columns=["question", "answer"])

# sanity check
print(train_ds[0]["conversations"])
print(eval_ds[0]["conversations"])


In [ ]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

train_ds = train_ds.map(formatting_prompts_func, batched=True)
eval_ds  = eval_ds.map(formatting_prompts_func, batched=True)

# verify formatting matches on both splits, and check where the
# assistant turn actually starts (should be "<|im_start|>assistant\n")
print(train_ds[0]["text"])
print("---")
print(eval_ds[0]["text"])


## WandB login

Uses `WANDB_API_KEY` from the environment — no hardcoded key.

In [ ]:
import wandb

wandb.login(key=os.environ["WANDB_API_KEY"])


## Training

Fixes applied:
- Model `dtype` (above) and trainer `fp16`/`bf16` flags below are both
  derived from the same `BF16_OK` check, so they can never disagree.
- `response_part` is `"<|im_start|>assistant\n"` (not
  `"<|im_end|>assistant\n"` — that variant never matches inside the
  tokenized text, which silently masks every label to -100, i.e. zero
  real training signal).
- Both `train_dataset`/`eval_dataset` already carry a pre-computed
  `"text"` field, so `dataset_text_field="text"` is used directly with
  no separate `formatting_func` passed to `SFTTrainer` (avoids
  double-formatting).
- HF push uses `os.environ["HF_TOKEN"]` — no literal token in the
  notebook. Set `HF_USERNAME` below to your Hugging Face username so
  the adapter lands in the right namespace.

In [ ]:
HF_USERNAME = "Srishtik"  # <-- change if this isn't your HF username


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
import matplotlib.pyplot as plt

def train_adapter(dataset, test_dataset, output_dir):
    wandb.init(
        project="qwen3-0.6B-trained-on-gsm8k",
        name=output_dir,
        config={
            "model": "qwen3-0.6b",
            "r": 16,
            "lora_alpha": 32,
            "max_steps": 1000,
            "learning_rate": 2e-4,
            "batch_size": 2,
            "gradient_accumulation_steps": 4,
            "warmup_steps": 5,
            "weight_decay": 0.001,
            "lr_scheduler": "linear",
            "max_seq_length": 2048,
        }
    )

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen3-0.6B",
        max_seq_length=2048,
        dtype=COMPUTE_DTYPE,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        use_rslora=False,
        loftq_config=None,
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        eval_dataset=test_dataset,
        args=SFTConfig(
            dataset_text_field="text",
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=5,
            max_steps=1000,
            learning_rate=2e-4,
            logging_steps=5,
            optim="adamw_8bit",
            weight_decay=0.001,
            lr_scheduler_type="linear",
            seed=3407,
            fp16=not BF16_OK,
            bf16=BF16_OK,
            report_to="wandb",
            padding_free=False,
            output_dir=output_dir,
            save_strategy="steps",
            save_steps=125,
            save_total_limit=5,
            eval_strategy="steps",
            eval_steps=20,
            max_grad_norm=1.0,
            logging_nan_inf_filter=False,
        )
    )

    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )

    trainer.train()

    log_history = trainer.state.log_history
    steps      = [e["step"] for e in log_history if "loss" in e and "eval_loss" not in e]
    losses     = [e["loss"] for e in log_history if "loss" in e and "eval_loss" not in e]
    grad_steps = [e["step"] for e in log_history if "grad_norm" in e]
    grad_norms = [e["grad_norm"] for e in log_history if "grad_norm" in e]
    eval_steps  = [e["step"] for e in log_history if "eval_loss" in e]
    eval_losses = [e["eval_loss"] for e in log_history if "eval_loss" in e]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    ax1.plot(steps, losses, color="steelblue", label="Train Loss")
    if eval_losses:
        ax1.plot(eval_steps, eval_losses, color="red",
                 linestyle="--", marker="o", label="Eval Loss")
    ax1.set_ylabel("Loss")
    ax1.set_title(f"Training Curves — {output_dir}")
    ax1.legend()
    ax1.grid(True)

    ax2.plot(grad_steps, grad_norms, color="darkorange")
    ax2.set_ylabel("Gradient Norm")
    ax2.set_xlabel("Steps")
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig(f"{output_dir}_curves.png")
    wandb.log({"training_curves": wandb.Image(f"{output_dir}_curves.png")})
    plt.show()
    wandb.finish()

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    hf_token = os.environ["HF_TOKEN"]
    repo_id = f"{HF_USERNAME}/{output_dir}"
    model.push_to_hub(repo_id, token=hf_token)
    tokenizer.push_to_hub(repo_id, token=hf_token)
    print(f"Pushed adapter to https://huggingface.co/{repo_id}")


In [ ]:
train_adapter(train_ds, eval_ds, "qwen3-0.6B-trained-on-gsm8k")
